# 框架运行时、数据与性能补充线 · 第 4/8 课：Collective 成本模型与拓扑选择

> 状态：**未开始**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现 ring/tree 的 α-β 粗模型，并解释为什么模型只能筛选、不能替代 nccl-tests。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/` 已计算通信量；本课把 latency α、带宽 β、消息大小和拓扑 hop 连接到 runtime algorithm 选择。

前置：Python、PyTorch、train 第 1～5 课、CUDA 基础。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Ring all-reduce 近似 `2(p-1)α + 2(p-1)/p·n/B`；tree 近似 `2log₂p·α + 2log₂p·n/B_tree`。真实 NCCL 会按拓扑、协议和 channel 调整。

### 数据与控制如何流动

把 world size、消息字节、候选算法的步骤数和实测有效带宽代入粗模型，先筛掉明显不合适的选择；再用目标拓扑上的 nccl-tests 与训练 trace 校准 α、B 和层级边界。

### 正确性条件与常见误区

B 必须是有效单流/路径带宽而非宣传总带宽；跨节点层级、共享 NIC、rail 和 PCIe root 会改变模型。collective 各 rank count/dtype 仍需满足合同。

### 性能、成本与工程取舍

大消息 ring 带宽利用好；小消息 tree 步数少。层级算法可先节点内 reduce、再跨节点、再广播，但增加阶段和调度复杂度。

## 具体演示

p=64、α=5μs 时 ring 延迟项 630μs，tree 约 60μs；消息足够大后带宽项可能反转选择。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐 ring/tree 粗略时间，并返回更快算法名。单位统一为秒、字节、字节/秒。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
import math

def choose_allreduce(world, nbytes, alpha_s, ring_bw, tree_bw):
    if world < 2 or nbytes < 0 or alpha_s < 0 or ring_bw <= 0 or tree_bw <= 0:
        raise ValueError("invalid model")
    ring = 2 * (world - 1) * alpha_s + 2 * (world - 1) / world * nbytes / ring_bw
    levels = math.ceil(math.log2(world))
    tree = 2 * levels * alpha_s + 2 * levels * nbytes / tree_bw
    # TODO：同时返回选择和两种估算，便于审计。
    return ______

name, ring_t, tree_t = choose_allreduce(64, 1024, 5e-6, 100e9, 100e9)
assert name == "tree" and tree_t < ring_t


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么把 NVLink 总双向带宽直接代入 B 会过度乐观？

**你的答案：**


### Q2

小消息 tree 更快的核心原因是什么？

**你的答案：**


### Q3

模型预测 ring 快，实测却慢，排查顺序是什么？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考资料

- [NCCL User Guide](https://docs.nvidia.com/deeplearning/nccl/user-guide/index.html)
- [PyTorch distributed](https://docs.pytorch.org/docs/stable/distributed.html)

API 与平台能力会演进；部署前应按目标版本重新核对。